# KV compression sweep — Kaggle runner

Six languages, equal-memory protocol. See `docs/PLAN.md` for what each run answers
and `docs/RUNBOOK.md` §9 for the full recipe.

**Before running:** Settings → Accelerator **GPU T4 ×2**, Internet **On**,
Add-ons → Secrets → `HF_TOKEN` (needed for the gated Llama weights).

One run = one shard. The sweep checkpoints after every document, so a 9-hour
timeout costs at most one document. Re-running with the same `OUT` and
`DOC_START` resumes.

In [ ]:
CONFIG = dict(
    MODEL="meta-llama/Llama-3.2-1B-Instruct",
    DTYPE="float32",          # Qwen-3B on T4: "float16"
    LANGS=["english", "nepali", "hindi", "bengali", "tamil", "amharic"],
    DOC_START=0,              # shard start: 0, 20, 40 ...
    DOCS=20,                  # shard size
    SENTS=10,
    MAX_TOKENS=1024,
    RATIOS=[0.3, 0.6, 0.8],
    METHODS=None,             # None = all ten policies
    NEEDLE=False,
    GEN_STEPS=0,
    PROTECT_LAST=0,
    OUT="out/compress_llama1b_s0.csv",
    BRANCH="paper/scale-up",
)
CONFIG

In [ ]:
import os, subprocess, sys

# Kaggle usually ships a recent torch/transformers; only add what is missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "scipy", "matplotlib"], check=False)

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", CONFIG["BRANCH"],
                    "https://github.com/LCfreak/llm-inference-test.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# HF token for the gated Llama weights (skip if you use an ungated model).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret:", e)

In [ ]:
# Parity gate: do not run experiments if this fails.
subprocess.run([sys.executable, "src/kv_compress.py", "--selftest",
                "--device", "cuda", "--dtype", "float32"], check=True)

In [ ]:
# Data + fertility table (internet on; cached afterwards).
subprocess.run([sys.executable, "src/make_data.py",
                "--tokenizer", CONFIG["MODEL"]], check=True)

In [ ]:
cmd = [sys.executable, "src/kv_compress.py",
       "--device", "cuda", "--dtype", CONFIG["DTYPE"],
       "--model", CONFIG["MODEL"],
       "--langs", *CONFIG["LANGS"],
       "--doc-start", str(CONFIG["DOC_START"]),
       "--docs", str(CONFIG["DOCS"]),
       "--sents", str(CONFIG["SENTS"]),
       "--max-tokens", str(CONFIG["MAX_TOKENS"]),
       "--ratios", *[str(r) for r in CONFIG["RATIOS"]],
       "--equal-memory", "--append",
       "--out", CONFIG["OUT"]]
if CONFIG["METHODS"]:
    cmd += ["--methods", *CONFIG["METHODS"]]
if CONFIG["NEEDLE"]:
    cmd += ["--needle"]
if CONFIG["GEN_STEPS"]:
    cmd += ["--gen-steps", str(CONFIG["GEN_STEPS"])]
if CONFIG["PROTECT_LAST"]:
    cmd += ["--protect-last", str(CONFIG["PROTECT_LAST"])]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Keep results in the version output so they survive the session.
os.makedirs("/kaggle/working/out", exist_ok=True)
for f in os.listdir("out"):
    if f.endswith(".csv"):
        subprocess.run(["cp", os.path.join("out", f), "/kaggle/working/out/"], check=True)
print(sorted(os.listdir("/kaggle/working/out")))

## Shard plan

| Run | `DOC_START` | `DOCS` | `OUT` |
|---|---:|---:|---|
| Llama-1B A | 0 | 20 | `out/compress_llama1b_s0.csv` |
| Llama-1B B | 20 | 20 | `out/compress_llama1b_s1.csv` |
| Llama-1B C | 40 | 20 | `out/compress_llama1b_s2.csv` |
| Qwen-3B A | 0 | 20 | `out/compress_qwen3b_s0.csv` |
| Qwen-3B B | 20 | 20 | `out/compress_qwen3b_s1.csv` |
| Qwen-3B C | 40 | 20 | `out/compress_qwen3b_s2.csv` |

Qwen-3B: `DTYPE="float16"`, `MODEL="Qwen/Qwen2.5-3B-Instruct"`.